# Explore the sampled dataset by tag

Interactive viewer for `dataset/manifests/manifest_all.json`. Pick one or more tags on the
left and the grid shows matching images with their predicted tags (the tags you selected are
marked with `*`). Choose a dataset, `any` vs `all` match, and how many images to show.

Same logic is available from the command line — e.g. `python3 viz_manifest.py --tags Roundabouts`.

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output
import viz_manifest as v

samples = v.load()                 # dataset/manifests/manifest_all.json
counts  = v.tag_counts(samples)
print(f'{len(samples):,} images · {len(counts)} tags seen · datasets: {v.datasets(samples)}')

9,298 images · 40 tags seen · datasets: ['bdd100k', 'culane', 'curvelanes', 'tusimple']


In [2]:
# ---- interactive tag picker ----
opts    = [f'{t}  ({c})' for t, c in counts.most_common()]
to_name = {f'{t}  ({c})': t for t, c in counts.most_common()}

tags_w = W.SelectMultiple(options=opts, description='Tags', rows=16,
                          layout=W.Layout(width='340px'))
ds_w   = W.Dropdown(options=['all'] + v.datasets(samples), description='Dataset')
mode_w = W.ToggleButtons(options=['any', 'all'], description='Match',
                         tooltips=['has ANY selected tag', 'has ALL selected tags'])
n_w    = W.IntSlider(value=12, min=1, max=48, description='Max imgs')
cols_w = W.IntSlider(value=4,  min=1, max=6,  description='Cols')
out    = W.Output()

def render(*_):
    with out:
        clear_output(wait=True)
        tags = [to_name[o] for o in tags_w.value]
        ds   = None if ds_w.value == 'all' else ds_w.value
        v.view(samples, tags=tags, mode=mode_w.value, dataset=ds,
               cols=cols_w.value, max_n=n_w.value)
        plt.show()

for w in (tags_w, ds_w, mode_w, n_w, cols_w):
    w.observe(render, names='value')

display(W.HBox([tags_w, W.VBox([ds_w, mode_w, n_w, cols_w])]), out)
render()

Output()

### One-liners (no widgets)
`v.view(samples, tags=['Nighttime', 'Glare'], mode='all', max_n=12)` — images with **both** tags.

`v.view(samples, tags=['Roundabouts'], dataset='bdd100k')` — filter to one dataset.

`v.view(samples, tags=['Crosswalks'], max_n=16, save='crosswalks.png')` — write a PNG.

In [ ]:
# example: every roundabout image across all datasets
v.view(samples, tags=['Roundabouts'], max_n=12)
plt.show()